# Dataset Overview: 4fqi_h1

This notebook examines the 4fqi_h1 mutant library: the mutation-count distribution,
the binding-score histogram (with the censoring floor at 7.0), the 16 binary variable
positions, and the structure-vs-library mismatches at indices 23 and 45.

In [ ]:
import sys
from pathlib import Path

LIBRARY_PATH = Path("../data/processed/4fqi_h1_library.parquet")
COMPANION_PATH = LIBRARY_PATH.with_suffix(".json")

for p, script in [(LIBRARY_PATH, "01_build_library.py"), (COMPANION_PATH, "01_build_library.py")]:
    if not p.exists():
        print(f"Missing: {p}")
        print(f"Run: python scripts/{script} --dataset 4fqi_h1")
        sys.exit(1)

print("All required artifacts found.")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb

df = pd.read_parquet(LIBRARY_PATH)
with open(COMPANION_PATH) as f:
    companion = json.load(f)

print(f"Rows: {len(df)}")
print(f"Reference sequence length: {len(companion['reference_seq'])}")
print(f"Variable positions: {companion['variable_positions']}")
print(f"Columns: {list(df.columns)}")

## Mutation-count distribution vs binomial expectation

With 16 binary variable positions the library is combinatorially complete:
the count of variants with exactly k mutations is C(16, k).

In [ ]:
n_pos = 16
observed = df["n_mut"].value_counts().sort_index()
expected_counts = {k: comb(n_pos, k) for k in range(n_pos + 1)}

fig, ax = plt.subplots(figsize=(8, 4))
ks = list(range(n_pos + 1))
ax.bar([k - 0.2 for k in ks], [observed.get(k, 0) for k in ks], width=0.4, label="Observed")
ax.bar([k + 0.2 for k in ks], [expected_counts[k] for k in ks], width=0.4, label="C(16, k)", alpha=0.7)
ax.set_xlabel("Number of mutations (n_mut)")
ax.set_ylabel("Count")
ax.set_title("Mutation-count distribution: observed vs C(16, k)")
ax.legend()
ax.set_xticks(ks)
plt.tight_layout()
plt.show()

total_expected = sum(expected_counts.values())
print(f"Total expected: 2^16 = {2**16} = {total_expected}")
print(f"Total observed: {len(df)}")

## Binding-score histogram

The binding score is `-log10(Kd)`-like, ranging from 7.000 to 9.835.
There is a censoring floor at exactly 7.000 (1,675 rows, 2.6%), meaning
variants below the detection limit are piled up at this value. Spearman
ties at the floor are expected and handled by the tie-corrected rank method.

In [ ]:
scores = df["binding_score"]
n_floor = (scores == 7.0).sum()
pct_floor = n_floor / len(df) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=80, edgecolor="black", linewidth=0.3)
ax.axvline(7.0, color="red", linestyle="--", linewidth=1.5, label=f"Censoring floor (n={n_floor}, {pct_floor:.1f}%)")
ax.set_xlabel("Binding score (-log10 Kd-like)")
ax.set_ylabel("Count")
ax.set_title("4fqi_h1 binding-score distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Score range: {scores.min():.3f} - {scores.max():.3f}")
print(f"Censoring floor at 7.000: {n_floor} rows ({pct_floor:.1f}%)")

## Variable positions and their two states

In [ ]:
alphabet_at = companion["alphabet_at"]
var_table = []
ref_seq = companion["reference_seq"]
for pos in companion["variable_positions"]:
    aas = alphabet_at[str(pos)]
    ref_aa = ref_seq[pos]
    mut_aa = [a for a in aas if a != ref_aa][0] if len(aas) == 2 else "?"
    var_table.append({"Position": pos, "Reference": ref_aa, "Mutant": mut_aa, "States": "/".join(aas)})

var_df = pd.DataFrame(var_table)
print(var_df.to_string(index=False))

## Structure-vs-library mismatches

The PDB H chain is index-aligned to the library (same length, no offset) but
differs at two **fixed** (non-variable) positions:
- Index 23: PDB has S, library has A
- Index 45: PDB has D, library has E

These are conservative substitutions (S/A are small; D/E are both acidic) and
occur at positions that do not vary across library members, so they affect the
embedding baseline but not the differential attribution.

In [ ]:
alignment = companion.get("structure_alignment", {})
if alignment:
    print(f"Same length: {alignment['same_length']}")
    print(f"Offset: {alignment['offset']}")
    print(f"Mismatches: {len(alignment['mismatches'])}")
    for idx, pdb_aa, lib_aa in alignment["mismatches"]:
        print(f"  Index {idx}: PDB={pdb_aa}, Library={lib_aa}")
else:
    print("No structure alignment data in companion JSON.")

## Conclusion

The 4fqi_h1 library contains 65,094 variants across 16 binary variable positions on
a 121-residue heavy chain, with a constant 109-residue light chain. The library is
combinatorially complete (2^16 = 65,536 possible; 65,094 observed, missing 442).
Binding scores range from 7.000 to 9.835, with 1,675 rows (2.6%) censored at the
detection floor of 7.000. The PDB structure aligns perfectly in length with two
conservative fixed-position mismatches at indices 23 (S vs A) and 45 (D vs E).